# 🔍 Suraksha - Pattern-Based Feature Engineering (Baseline Solution)

## Objective
Engineer **pattern-based features** from transaction-level data **WITHOUT user tracking**.

## Baseline Solution Approach
* **Input**: Official dataset from `workspace.bronze.upi`
* **Output**: `workspace.silver.upi_pattern_features`
* **No user identifiers**: Works without VPAs, device IDs, or behavioral tracking
* **4 Fraud Types**:
  1. **Amount Anomaly** - Statistical outliers in transaction amounts
  2. **Temporal Anomaly** - Unusual timing patterns (odd hours, weekends)
  3. **Merchant Fraud** - High-risk merchant categories
  4. **High-Risk Pattern** - Combination of suspicious indicators

## Key Difference from Advanced Solution
* ❌ No sender_vpa, receiver_vpa, device_id
* ❌ No velocity tracking (requires user history)
* ❌ No behavioral features (requires user identifiers)
* ✅ Only transaction-level patterns
* ✅ Statistical anomaly detection
* ✅ Temporal pattern analysis
* ✅ Merchant risk scoring

In [0]:
# Import required libraries
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, when, hour, dayofweek, month, year,
    mean, stddev, percentile_approx, count, sum as _sum,
    lag, lead, datediff, abs as _abs,
    lit, concat_ws, expr, udf
)
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🔍 SURAKSHA - PATTERN-BASED FEATURE ENGINEERING")
print("="*70)

# Configuration - BASELINE SOLUTION
source_table = "workspace.bronze.upi_official"  # ← Official data (baseline)
target_table = "workspace.silver.upi_pattern_features"

print(f"\n📋 Configuration:")
print(f"   Solution: BASELINE")
print(f"   Source: {source_table}")
print(f"   Target: {target_table}")
print(f"   Approach: Pattern-based (NO user tracking)")
print(f"\n✓ Configuration loaded")

In [0]:
# Load data from bronze layer
print("Loading data from bronze layer...")
df_bronze = spark.read.table(source_table)

print(f"✓ Loaded {df_bronze.count():,} transactions")
print(f"\nSchema:")
df_bronze.printSchema()

print(f"\nSample data:")
display(df_bronze.limit(5))

In [0]:
# Extract temporal features (no user tracking needed)
print("Engineering temporal pattern features...")

df_temporal = df_bronze.withColumn("hour", hour(col("timestamp"))) \
    .withColumn("day_of_week", dayofweek(col("timestamp"))) \
    .withColumn("month", month(col("timestamp"))) \
    .withColumn("year", year(col("timestamp")))

# Define temporal anomaly patterns
df_temporal = df_temporal \
    .withColumn("is_odd_hours", 
                when((col("hour") >= 23) | (col("hour") <= 6), True)
                .otherwise(False)) \
    .withColumn("is_weekend", 
                when(col("day_of_week").isin([1, 7]), True)  # Sunday=1, Saturday=7
                .otherwise(False)) \
    .withColumn("is_late_night", 
                when((col("hour") >= 2) & (col("hour") <= 5), True)
                .otherwise(False)) \
    .withColumn("is_business_hours", 
                when((col("hour") >= 9) & (col("hour") <= 17), True)
                .otherwise(False))

print("✓ Temporal pattern features created")
print(f"  - is_odd_hours (11 PM - 6 AM)")
print(f"  - is_weekend (Saturday/Sunday)")
print(f"  - is_late_night (2 AM - 5 AM)")
print(f"  - is_business_hours (9 AM - 5 PM)")

In [0]:
# Calculate statistical features for amount anomalies
print("\nEngineering amount statistical features...")

# Calculate global statistics
amount_stats = df_temporal.select(
    mean("amount_inr").alias("global_mean"),
    stddev("amount_inr").alias("global_stddev"),
    percentile_approx("amount_inr", 0.25).alias("q25"),
    percentile_approx("amount_inr", 0.50).alias("median"),
    percentile_approx("amount_inr", 0.75).alias("q75"),
    percentile_approx("amount_inr", 0.90).alias("p90"),
    percentile_approx("amount_inr", 0.95).alias("p95"),
    percentile_approx("amount_inr", 0.99).alias("p99")
).collect()[0]

global_mean = amount_stats["global_mean"]
global_stddev = amount_stats["global_stddev"]
q25 = amount_stats["q25"]
median = amount_stats["median"]
q75 = amount_stats["q75"]
p90 = amount_stats["p90"]
p95 = amount_stats["p95"]
p99 = amount_stats["p99"]

print(f"✓ Global statistics calculated:")
print(f"  Mean: ₹{global_mean:,.2f}")
print(f"  Std Dev: ₹{global_stddev:,.2f}")
print(f"  95th percentile: ₹{p95:,.2f}")
print(f"  99th percentile: ₹{p99:,.2f}")

# Add amount anomaly features
df_amount = df_temporal \
    .withColumn("amount_zscore", 
                (col("amount_inr") - lit(global_mean)) / lit(global_stddev)) \
    .withColumn("is_high_amount", 
                when(col("amount_inr") >= lit(p95), True).otherwise(False)) \
    .withColumn("is_very_high_amount", 
                when(col("amount_inr") >= lit(p99), True).otherwise(False)) \
    .withColumn("is_round_amount", 
                when(col("amount_inr") % 1000 == 0, True).otherwise(False)) \
    .withColumn("amount_category",
                when(col("amount_inr") < lit(q25), "low")
                .when(col("amount_inr") < lit(median), "medium_low")
                .when(col("amount_inr") < lit(q75), "medium_high")
                .when(col("amount_inr") < lit(p90), "high")
                .otherwise("very_high"))

print("✓ Amount anomaly features created")
print(f"  - amount_zscore (z-score from global mean)")
print(f"  - is_high_amount (>= 95th percentile)")
print(f"  - is_very_high_amount (>= 99th percentile)")
print(f"  - is_round_amount (divisible by 1000)")
print(f"  - amount_category (low to very_high)")

In [0]:
# Define high-risk merchant categories
print("\nEngineering merchant risk features...")

# High-risk merchant categories based on fraud patterns
high_risk_categories = ['Entertainment', 'Electronics', 'Jewelry', 'Gambling', 'Adult']
medium_risk_categories = ['Shopping', 'Travel', 'Luxury']

df_merchant = df_amount \
    .withColumn("is_high_risk_merchant",
                when(col("merchant_category").isin(high_risk_categories), True)
                .otherwise(False)) \
    .withColumn("is_medium_risk_merchant",
                when(col("merchant_category").isin(medium_risk_categories), True)
                .otherwise(False)) \
    .withColumn("is_p2m",
                when(col("txn_type") == "P2M", True).otherwise(False)) \
    .withColumn("merchant_risk_score",
                when(col("merchant_category").isin(high_risk_categories), 3)
                .when(col("merchant_category").isin(medium_risk_categories), 2)
                .when(col("txn_type") == "P2M", 1)
                .otherwise(0))

print("✓ Merchant risk features created")
print(f"  High-risk categories: {high_risk_categories}")
print(f"  Medium-risk categories: {medium_risk_categories}")
print(f"  Features: is_high_risk_merchant, merchant_risk_score")

In [0]:
# Create transaction pattern features (no user tracking)
print("\nEngineering transaction pattern features...")

df_patterns = df_merchant \
    .withColumn("failed_txn",
                when(col("txn_status") == "FAILED", 1).otherwise(0)) \
    .withColumn("large_round_amount",
                when((col("is_round_amount") == True) & (col("amount_inr") >= 10000), True)
                .otherwise(False)) \
    .withColumn("odd_hour_high_amount",
                when((col("is_odd_hours") == True) & (col("is_high_amount") == True), True)
                .otherwise(False)) \
    .withColumn("weekend_high_amount",
                when((col("is_weekend") == True) & (col("is_high_amount") == True), True)
                .otherwise(False)) \
    .withColumn("high_risk_merchant_high_amount",
                when((col("is_high_risk_merchant") == True) & (col("is_high_amount") == True), True)
                .otherwise(False))

print("✓ Transaction pattern features created")
print(f"  - failed_txn (FAILED status indicator)")
print(f"  - large_round_amount (round + high value)")
print(f"  - odd_hour_high_amount (suspicious timing + amount)")
print(f"  - weekend_high_amount (weekend + high amount)")
print(f"  - high_risk_merchant_high_amount (risky merchant + high amount)")

In [0]:
# Calculate composite risk score for pattern-based detection
print("\nCalculating composite risk scores...")

df_risk = df_patterns \
    .withColumn("temporal_risk",
                (col("is_odd_hours").cast("int") * 2) +
                (col("is_late_night").cast("int") * 3) +
                (col("is_weekend").cast("int") * 1)) \
    .withColumn("amount_risk",
                (col("is_high_amount").cast("int") * 2) +
                (col("is_very_high_amount").cast("int") * 3) +
                (col("is_round_amount").cast("int") * 1) +
                when(_abs(col("amount_zscore")) > 3, 3)
                .when(_abs(col("amount_zscore")) > 2, 2)
                .otherwise(0)) \
    .withColumn("pattern_risk",
                (col("failed_txn") * 2) +
                (col("large_round_amount").cast("int") * 2) +
                (col("odd_hour_high_amount").cast("int") * 3) +
                (col("weekend_high_amount").cast("int") * 1) +
                (col("high_risk_merchant_high_amount").cast("int") * 3)) \
    .withColumn("total_risk_score",
                col("temporal_risk") + 
                col("amount_risk") + 
                col("merchant_risk_score") +
                col("pattern_risk"))

print("✓ Composite risk scores calculated")
print(f"  - temporal_risk (0-6 points)")
print(f"  - amount_risk (0-9 points)")
print(f"  - pattern_risk (0-11 points)")
print(f"  - total_risk_score (sum of all risk components)")

# Show risk score distribution
print("\nRisk score distribution:")
risk_dist = df_risk.groupBy("total_risk_score").count().orderBy("total_risk_score")
display(risk_dist)

In [0]:
# Create pattern-based fraud type labels (4 types for baseline)
print("\nCreating pattern-based fraud type labels...")

df_labeled = df_risk \
    .withColumn("pattern_fraud_type",
                # Priority-based classification
                when((col("is_very_high_amount") == True) | (col("amount_zscore") > 3),
                     "Amount Anomaly")
                .when((col("is_late_night") == True) | (col("odd_hour_high_amount") == True),
                     "Temporal Anomaly")
                .when((col("high_risk_merchant_high_amount") == True),
                     "Merchant Fraud")
                .when(col("total_risk_score") >= 8,
                     "High-Risk Pattern")
                .otherwise("Legitimate")) \
    .withColumn("pattern_fraud_type_id",
                when(col("pattern_fraud_type") == "Legitimate", 0)
                .when(col("pattern_fraud_type") == "Amount Anomaly", 1)
                .when(col("pattern_fraud_type") == "Temporal Anomaly", 2)
                .when(col("pattern_fraud_type") == "Merchant Fraud", 3)
                .when(col("pattern_fraud_type") == "High-Risk Pattern", 4)
                .otherwise(0)) \
    .withColumn("is_pattern_fraud",
                when(col("pattern_fraud_type") != "Legitimate", 1).otherwise(0))

print("✓ Pattern-based fraud labels created")
print("\nFraud Type Mapping:")
print("  0 - Legitimate")
print("  1 - Amount Anomaly")
print("  2 - Temporal Anomaly")
print("  3 - Merchant Fraud")
print("  4 - High-Risk Pattern")

# Show distribution
print("\nPattern-based fraud type distribution:")
fraud_dist = df_labeled.groupBy("pattern_fraud_type", "is_pattern_fraud").count().orderBy("count", ascending=False)
display(fraud_dist)

In [0]:
# Save engineered features to silver layer
print("\nSaving pattern features to silver layer...")
print(f"Target table: {target_table}")

# Create schema if needed
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

# Write to Delta Lake
df_labeled.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"✓ Data successfully saved to {target_table}")
print(f"\nTable location: {spark.sql(f'DESCRIBE DETAIL {target_table}').select('location').collect()[0][0]}")

In [0]:
# Verify the saved data
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

df_verify = spark.read.table(target_table)

print(f"\n1. Row count: {df_verify.count():,}")

print(f"\n2. Pattern fraud distribution:")
pattern_dist = df_verify.groupBy("pattern_fraud_type", "is_pattern_fraud").count().orderBy("count", ascending=False)
display(pattern_dist)

print(f"\n3. Risk score statistics:")
risk_stats = df_verify.select(
    mean("total_risk_score").alias("avg_risk"),
    stddev("total_risk_score").alias("stddev_risk"),
    percentile_approx("total_risk_score", 0.50).alias("median_risk"),
    percentile_approx("total_risk_score", 0.95).alias("p95_risk")
)
display(risk_stats)

print(f"\n4. Feature summary:")
feature_cols = ["is_odd_hours", "is_high_amount", "is_high_risk_merchant", 
                "temporal_risk", "amount_risk", "pattern_risk", "total_risk_score"]
for col_name in feature_cols:
    # Cast boolean columns to int for comparison
    count_val = df_verify.filter(col(col_name).cast("int") > 0).count()
    print(f"  {col_name}: {count_val:,} transactions")

print(f"\n5. Sample records:")
display(df_verify.select(
    "txn_id", "timestamp", "amount_inr", "pattern_fraud_type", 
    "total_risk_score", "is_pattern_fraud"
).limit(10))

print(f"\n" + "="*80)
print("✓ PATTERN FEATURE ENGINEERING COMPLETE")
print("="*80)
print(f"\nTable: {target_table}")
print(f"Total rows: {df_verify.count():,}")
print(f"Pattern-based fraud: {df_verify.filter('is_pattern_fraud = 1').count():,}")
print(f"Legitimate: {df_verify.filter('is_pattern_fraud = 0').count():,}")
print(f"\n▶ Ready for model training in notebook 03_binary_training")

## ✓ Pattern Feature Engineering Summary

### Features Created (No User Tracking)

**1. Temporal Pattern Features:**
* `is_odd_hours` - Transactions between 11 PM - 6 AM
* `is_weekend` - Saturday/Sunday transactions
* `is_late_night` - 2 AM - 5 AM (highest risk)
* `is_business_hours` - 9 AM - 5 PM

**2. Amount Statistical Features:**
* `amount_zscore` - Statistical deviation from mean
* `is_high_amount` - Above 95th percentile
* `is_very_high_amount` - Above 99th percentile
* `is_round_amount` - Divisible by 1000
* `amount_category` - low/medium_low/medium_high/high/very_high

**3. Merchant Risk Features:**
* `is_high_risk_merchant` - Entertainment, Electronics, Jewelry
* `is_medium_risk_merchant` - Shopping, Travel, Luxury
* `merchant_risk_score` - 0-3 risk points

**4. Composite Pattern Features:**
* `large_round_amount` - Round numbers + high value
* `odd_hour_high_amount` - Suspicious timing + amount
* `weekend_high_amount` - Weekend + high amount
* `high_risk_merchant_high_amount` - Risky merchant + high amount

**5. Risk Scores:**
* `temporal_risk` - Time-based risk (0-6 points)
* `amount_risk` - Amount-based risk (0-9 points)
* `pattern_risk` - Pattern-based risk (0-11 points)
* `total_risk_score` - Combined risk score

### Pattern-Based Fraud Types (4 Types)

1. **Amount Anomaly** (Type 1) - Statistical outliers
2. **Temporal Anomaly** (Type 2) - Odd hours, late night
3. **Merchant Fraud** (Type 3) - High-risk merchants
4. **High-Risk Pattern** (Type 4) - Multiple risk indicators

### Output
* **Table**: `workspace.silver.upi_pattern_features`
* **Format**: Delta Lake
* **Ready for**: Binary classification model training

### Next Steps
1. ▶ Train binary classification model (03_binary_training)
2. ▶ Register model as "suraksha_baseline" in MLflow
3. ▶ Compare with advanced solution (9 fraud types)